# DBpedia Ontology Dataset - Text Classification

The DBpedia Ontology Dataset is commonly used for text classification tasks in natural language processing (NLP). It is derived from DBpedia, a structured knowledge base that extracts information from Wikipedia. In the context of text classification, the dataset provides short text samples (like titles or abstracts) along with their corresponding categories (ontology classes from DBpedia).

Each data sample consists of:

    - A short text (usually a title and description of a Wikipedia article)

    - A label (DBpedia ontology class)

Here are examples of the 14 DBpedia classes:

    - Company
    - EducationalInstitution
    - Artist
    - Athlete
    - OfficeHolder
    - MeanOfTransportation
    - Building
    - NaturalPlace
    - Village
    - Animal
    - Plant
    - Album
    - Film
    - WrittenWork

## Remove existing torch-related packages

In [ ]:
!pip uninstall -y torch torchvision torchaudio  # Remove existing torch-related packages

## Install packages

In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy

## Ensure latest datasets version

In [ ]:
!pip install --upgrade datasets  # Ensure latest datasets version

# Imports

In [2]:
# Model Implementation and Fine Tuning - DBpedia Ontology Classification

# The DBpedia dataset contains text classified into ontology classes.
# This script compares transformer models for text classification on DBpedia.

### Acceptance Criterion
# The final output should be a report comparing:
# - Performance (accuracy, F1-score),
# - Efficiency (training/inference time),
# - Resource usage (model size, memory usage) of selected transformer models.

## Imports
#!pip install transformers datasets torch pandas

## Import necessary libraries
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    GPT2Tokenizer, GPT2ForSequenceClassification
)
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import numpy as np
import os
import zipfile

# Device Configuration

In [ ]:
## Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Dataset from Google Drive

In [ ]:
## Load Dataset
# Load DBpedia train and test CSV files
train_df = pd.read_csv("/content/drive/MyDrive/dbpedia_data/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/dbpedia_data/test.csv")

# Load Dataset from Local

In [4]:
## Load Dataset
# Load DBpedia train and test CSV files
train_df = pd.read_csv("DBpedia_Classification/train.csv")
test_df = pd.read_csv("DBpedia_Classification/test.csv")

In [5]:
train_df.head()

,label,title,content
0,0,E. D. Abbott Ltd,Abbott of Farnham E D Abbott Limited was a Br...
1,0,Schwan-Stabilo,Schwan-STABILO is a German maker of pens for ...
2,0,Q-workshop,Q-workshop is a Polish company located in Poz...
3,0,Marvell Software Solutions Israel,Marvell Software Solutions Israel known as RA...
4,0,Bergan Mercy Medical Center,Bergan Mercy Medical Center is a hospital loc...


In [7]:
train_df.size

1680000

In [8]:
test_df.size

210000

## Display first few rows to inspect data

In [ ]:
# Display first few rows to inspect data
print("Training data sample:")
print(train_df.head())
print(f"Training data size: {train_df.size}")
print(f"Test data size: {test_df.size}")
print("Unique labels in train:", train_df["label"].unique())

Training data sample:
   label                              title  \
0      0                   E. D. Abbott Ltd   
1      0                     Schwan-Stabilo   
2      0                         Q-workshop   
3      0  Marvell Software Solutions Israel   
4      0        Bergan Mercy Medical Center   

                                             content  
0   Abbott of Farnham E D Abbott Limited was a Br...  
1   Schwan-STABILO is a German maker of pens for ...  
2   Q-workshop is a Polish company located in Poz...  
3   Marvell Software Solutions Israel known as RA...  
4   Bergan Mercy Medical Center is a hospital loc...  
Training data size: 1680000
Test data size: 210000
Unique labels in train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]


## Process Data

In [ ]:
## Process Data
# Combine 'title' and 'content' into a single 'text' column
train_df["text"] = train_df["title"] + " " + train_df["content"]
test_df["text"] = test_df["title"] + " " + test_df["content"]

# Labels are already 0-based, no adjustment needed
train_df["label"] = train_df["label"]
test_df["label"] = test_df["label"]

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

# Create a DatasetDict
datasets = DatasetDict({"train": train_dataset, "test": test_dataset})

# Tokenization & Evaluation Functions

In [ ]:
# Tokenization Function
def tokenize_function(examples, tokenizer, max_length=512):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

# Custom Collate Function
# to process batches of data for the DataLoader in the DBpedia ontology classification task.
def custom_collate_fn(batch):
    input_ids = torch.stack([torch.tensor(item["input_ids"], dtype=torch.long) for item in batch])
    attention_mask = torch.stack([torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch])
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "label": labels
    }

# Evaluation Function
def evaluate_model(model, dataloader, device, model_name):
    model.eval()
    predictions = []
    true_labels = []
    inference_times = []
    memory_usages = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {model_name}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            labels = batch["label"].to(device)
            start_time = time.time()
            outputs = model(**inputs)
            inference_times.append(time.time() - start_time)
            batch_predictions = torch.argmax(outputs.logits, dim=-1)
            predictions.extend(batch_predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            if device.type == "cuda":
                memory_usages.append(torch.cuda.memory_allocated(device) / 1024**2)

    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average="weighted")
    recall = recall_score(true_labels, predictions, average="weighted")
    f1 = f1_score(true_labels, predictions, average="weighted")
    avg_inference_time = np.mean(inference_times)
    total_inference_time = np.sum(inference_times)
    avg_memory_usage = np.mean(memory_usages) if memory_usages else None

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_inference_time": avg_inference_time,
        "total_inference_time": total_inference_time,
        "avg_memory_usage": avg_memory_usage
    }

## Dictionary to store results

In [ ]:
# Dictionary to store results
results = {}

# RoBERTa Model

## Fine Tune Model

In [ ]:
# RoBERTa Model
print("Processing RoBERTa model...")
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=len(train_df["label"].unique())).to(device)
roberta_datasets = datasets.map(
    lambda x: tokenize_function(x, roberta_tokenizer, max_length=512),
    batched=True
)
roberta_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
roberta_train_loader = DataLoader(roberta_datasets["train"], batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
roberta_test_loader = DataLoader(roberta_datasets["test"], batch_size=16, collate_fn=custom_collate_fn)

optimizer = AdamW(roberta_model.parameters(), lr=2e-5)
num_epochs = 3
roberta_model.train()
roberta_training_times = []
for epoch in range(num_epochs):
    start_time = time.time()
    for batch in tqdm(roberta_train_loader, desc=f"RoBERTa Epoch {epoch+1}"):
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
        labels = batch["label"].to(device)
        outputs = roberta_model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    roberta_training_times.append(time.time() - start_time)

## Save model and tokenizer

In [ ]:
# Save model and tokenizer
roberta_output_dir = "/content/drive/MyDrive/dbpedia_results/roberta"
roberta_model.save_pretrained(roberta_output_dir)
roberta_tokenizer.save_pretrained(roberta_output_dir)
print(f"RoBERTa model and tokenizer saved to {roberta_output_dir}")

# Clear model from memory to free GPU resources
del roberta_model
torch.cuda.empty_cache()

## Load model and tokenizer from saved files

In [ ]:
# Load model and tokenizer from saved files
print("Loading RoBERTa model and tokenizer from saved files...")
roberta_tokenizer = RobertaTokenizer.from_pretrained(roberta_output_dir)
roberta_model = RobertaForSequenceClassification.from_pretrained(roberta_output_dir).to(device)

## Evaluate loaded model

In [ ]:
# Evaluate loaded model
roberta_results = evaluate_model(roberta_model, roberta_test_loader, device, "RoBERTa")
roberta_results["training_time"] = np.sum(roberta_training_times)
roberta_results["model_size_mb"] = sum(p.numel() for p in roberta_model.parameters()) * 4 / 1024**2
results["RoBERTa"] = roberta_results
print("RoBERTa evaluation completed.")

# DistilBERT Model

## Fine Tune Model

In [ ]:
# DistilBERT Model
print("Processing DistilBERT model...")
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
distilbert_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(train_df["label"].unique())).to(device)
distilbert_datasets = datasets.map(
    lambda x: tokenize_function(x, distilbert_tokenizer, max_length=256),
    batched=True
)
distilbert_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
distilbert_train_loader = DataLoader(distilbert_datasets["train"], batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
distilbert_test_loader = DataLoader(distilbert_datasets["test"], batch_size=16, collate_fn=custom_collate_fn)

optimizer = AdamW(distilbert_model.parameters(), lr=2e-5)
num_epochs = 3
distilbert_model.train()
distilbert_training_times = []
for epoch in range(num_epochs):
    start_time = time.time()
    for batch in tqdm(distilbert_train_loader, desc=f"DistilBERT Epoch {epoch+1}"):
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
        labels = batch["label"].to(device)
        outputs = distilbert_model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    distilbert_training_times.append(time.time() - start_time)



## Save model and tokenizer

In [ ]:
# Save model and tokenizer
distilbert_output_dir = "/content/drive/MyDrive/dbpedia_results/distilbert"
distilbert_model.save_pretrained(distilbert_output_dir)
distilbert_tokenizer.save_pretrained(distilbert_output_dir)
print(f"DistilBERT model and tokenizer saved to {distilbert_output_dir}")

# Clear model from memory to free GPU resources
del distilbert_model
torch.cuda.empty_cache()

## Load model and tokenizer from saved files

In [ ]:
# Load model and tokenizer from saved files
print("Loading DistilBERT model and tokenizer from saved files...")
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(distilbert_output_dir)
distilbert_model = DistilBertForSequenceClassification.from_pretrained(distilbert_output_dir).to(device)

## Evaluate loaded model

In [ ]:
# Evaluate loaded model
distilbert_results = evaluate_model(distilbert_model, distilbert_test_loader, device, "DistilBERT")
distilbert_results["training_time"] = np.sum(distilbert_training_times)
distilbert_results["model_size_mb"] = sum(p.numel() for p in distilbert_model.parameters()) * 4 / 1024**2
results["DistilBERT"] = distilbert_results
print("DistilBERT evaluation completed.")

# GPT-2 Model

In [ ]:
# GPT-2 Model
print("Processing GPT-2 model...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=len(train_df["label"].unique())).to(device)
gpt2_model.config.pad_token_id = gpt2_tokenizer.eos_token_id
gpt2_datasets = datasets.map(
    lambda x: tokenize_function(x, gpt2_tokenizer, max_length=512),
    batched=True
)
gpt2_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
gpt2_train_loader = DataLoader(gpt2_datasets["train"], batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
gpt2_test_loader = DataLoader(gpt2_datasets["test"], batch_size=16, collate_fn=custom_collate_fn)

optimizer = AdamW(gpt2_model.parameters(), lr=2e-5)
num_epochs = 3
gpt2_model.train()
gpt2_training_times = []
for epoch in range(num_epochs):
    start_time = time.time()
    for batch in tqdm(gpt2_train_loader, desc=f"GPT-2 Epoch {epoch+1}"):
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
        labels = batch["label"].to(device)
        outputs = gpt2_model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    gpt2_training_times.append(time.time() - start_time)


## Save model and tokenizer

In [ ]:
# Save model and tokenizer
gpt2_output_dir = "/content/drive/MyDrive/dbpedia_results/gpt2"
gpt2_model.save_pretrained(gpt2_output_dir)
gpt2_tokenizer.save_pretrained(gpt2_output_dir)
print(f"GPT-2 model and tokenizer saved to {gpt2_output_dir}")

# Clear model from memory to free GPU resources
del gpt2_model
torch.cuda.empty_cache()

## Load model and tokenizer from saved files

In [ ]:
# Load model and tokenizer from saved files
print("Loading GPT-2 model and tokenizer from saved files...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(gpt2_output_dir)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = GPT2ForSequenceClassification.from_pretrained(gpt2_output_dir).to(device)
gpt2_model.config.pad_token_id = gpt2_tokenizer.eos_token_id

## Evaluate loaded model

In [ ]:
# Evaluate loaded model
gpt2_results = evaluate_model(gpt2_model, gpt2_test_loader, device, "GPT-2")
gpt2_results["training_time"] = np.sum(gpt2_training_times)
gpt2_results["model_size_mb"] = sum(p.numel() for p in gpt2_model.parameters()) * 4 / 1024**2
results["GPT-2"] = gpt2_results
print("GPT-2 evaluation completed.")

# Save Results

In [ ]:
# Save Results
results_df = pd.DataFrame(results).T
results_df.to_csv("/content/drive/MyDrive/dbpedia_results/model_comparison.csv")

# Create zip archive
with zipfile.ZipFile("/content/drive/MyDrive/dbpedia_results/results.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("/content/drive/MyDrive/dbpedia_results/model_comparison.csv")

print("Results saved to /content/drive/MyDrive/dbpedia_results")